# Odometry Exercise 3 — plotting rotation data

Load a timestamped rotation log, plot the signed encoder counts and local odometry heading, and compare final reported rotation with your independent physical-angle measurements.

This notebook is an introduction to plotting odometry data. Start with the supplied synthetic example so that you can see what each cell produces. The example is not evidence about your robot. When you are ready, change the settings in the **Use your own data** cell and run the notebook again.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", context="notebook")
rng = np.random.default_rng(2026)


## 1. Use the example or your own data

Leave `USE_EXAMPLE_DATA` set to `True` on your first run. To use your measurements, upload the CSV exported from the Serial Monitor, set `USE_EXAMPLE_DATA = False`, and enter its filename. This is the main cell you need to edit.


In [ ]:
USE_EXAMPLE_DATA = True
CSV_FILENAME = "odometry_exercise03_samples.csv"

print("Using:", "synthetic example" if USE_EXAMPLE_DATA else CSV_FILENAME)


## 2. Create the small synthetic example

Run this cell when using the example. You do not need to understand or edit the generation code. Its columns match the rotation log in Exercise 3.


In [ ]:
example_rows = []
example_conditions = [
    (1, np.pi / 2, 1.48, 2.0),
    (2, -np.pi / 2, -1.50, 1.5),
    (3, np.pi, 2.98, 3.0),
    (4, -np.pi, -3.01, 2.5),
]

for trial_num, command_rad, final_heading_rad, centre_motion_mm in example_conditions:
    for sample_num, progress in enumerate(np.linspace(0, 1, 31)):
        turn_sign = np.sign(final_heading_rad)
        wheel_count = progress * abs(final_heading_rad) * 170
        example_rows.append({
            "test_name": "rotation",
            "trial_num": trial_num,
            "commanded_angle_rad": command_rad,
            "sample_time_ms": 1000 * trial_num + 50 * sample_num,
            "left_encoder_count": round(-turn_sign * wheel_count),
            "right_encoder_count": round(turn_sign * wheel_count),
            "odometry_x_mm": centre_motion_mm * np.sin(np.pi * progress),
            "odometry_y_mm": centre_motion_mm * np.sin(2 * np.pi * progress) / 2,
            "odometry_theta_rad": progress * final_heading_rad,
        })

example_data = pd.DataFrame(example_rows)


## 3. Load and preview the selected data

When `USE_EXAMPLE_DATA` is false, `pd.read_csv(...)` reads your file. The final line displays its first five rows.


In [ ]:
if USE_EXAMPLE_DATA:
    data = example_data.copy()
else:
    data = pd.read_csv(CSV_FILENAME)

data.head()


## 4. Calculate elapsed time

Subtract the first timestamp in each trial so that every plot begins at zero seconds. The heading is also converted to degrees for plotting.


In [ ]:
data = data.loc[data["test_name"] == "rotation"].copy()
data = data.sort_values(["trial_num", "sample_time_ms"])
data["elapsed_s"] = (
    data["sample_time_ms"]
    - data.groupby("trial_num")["sample_time_ms"].transform("first")
) / 1000
data["heading_deg"] = np.degrees(data["odometry_theta_rad"])
data["trial_label"] = "Trial " + data["trial_num"].astype(str)

data[["trial_num", "sample_time_ms", "elapsed_s", "heading_deg"]].head()


## 5. Plot both encoder counts

An in-place rotation should produce encoder-count changes with opposite signs. The plot lets you see how the two recorded count sequences developed during each turn.


In [ ]:
encoder_plot_data = data.melt(
    id_vars=["trial_num", "trial_label", "elapsed_s"],
    value_vars=["left_encoder_count", "right_encoder_count"],
    var_name="wheel",
    value_name="encoder_count",
)

grid = sns.relplot(
    data=encoder_plot_data,
    x="elapsed_s",
    y="encoder_count",
    hue="wheel",
    col="trial_label",
    col_wrap=2,
    kind="line",
    marker="o",
    estimator=None,
    height=3.2,
)
grid.set_axis_labels("Elapsed time (s)", "Encoder count")
grid.set_titles("{col_name}")
grid.figure.suptitle("Encoder counts during each rotation", y=1.03)
plt.show()


## 6. Plot the reported heading

This plot shows the heading accumulated by the local odometry model. It does not show the independently measured physical rotation.


In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
sns.lineplot(
    data=data,
    x="elapsed_s",
    y="heading_deg",
    hue="trial_label",
    marker="o",
    estimator=None,
    ax=ax,
)
ax.axhline(0, color="black", linewidth=1, linestyle="--")
ax.set(
    title="Heading reported by local odometry",
    xlabel="Elapsed time (s)",
    ylabel="Odometry heading (degrees)",
)
plt.show()


## 7. Plot reported movement of the robot centre

For an ideal rotation on the spot, x and y would remain unchanged. This is still only the local model's report, so compare it with what you observed physically.


In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
sns.lineplot(
    data=data,
    x="odometry_x_mm",
    y="odometry_y_mm",
    hue="trial_label",
    marker="o",
    estimator=None,
    ax=ax,
)
ax.set(
    title="Centre movement reported during rotation",
    xlabel="Odometry x (mm)",
    ylabel="Odometry y (mm)",
)
plt.show()


## 8. Extract the initial and final reported headings

The following cell creates one row per trial and calculates the change between its first and final odometry headings and centre positions.


In [ ]:
trial_summary = (
    data.groupby("trial_num", as_index=False)
    .agg(
        commanded_angle_rad=("commanded_angle_rad", "first"),
        initial_x_mm=("odometry_x_mm", "first"),
        initial_y_mm=("odometry_y_mm", "first"),
        initial_heading_rad=("odometry_theta_rad", "first"),
        final_heading_rad=("odometry_theta_rad", "last"),
        final_x_mm=("odometry_x_mm", "last"),
        final_y_mm=("odometry_y_mm", "last"),
    )
)
trial_summary["reported_angle_rad"] = (
    trial_summary["final_heading_rad"]
    - trial_summary["initial_heading_rad"]
)
trial_summary["reported_angle_deg"] = np.degrees(
    trial_summary["reported_angle_rad"]
)
trial_summary["reported_centre_change_x_mm"] = (
    trial_summary["final_x_mm"] - trial_summary["initial_x_mm"]
)
trial_summary["reported_centre_change_y_mm"] = (
    trial_summary["final_y_mm"] - trial_summary["initial_y_mm"]
)

trial_summary


## 9. Enter your physical angle measurements

The robot log does not contain the angle you measured on the printed sheet. Enter the independently measured start and final headings below in degrees. Use the same positive and negative convention as the odometry. Add or remove rows so that the trial numbers match your log. Record the radius and separation used by the model during each trial, and label the row as `calibration` or `held-back`.


In [ ]:
if USE_EXAMPLE_DATA:
    physical_measurements = pd.DataFrame({
        "trial_num": [1, 2, 3, 4],
        "measured_start_heading_deg": [0.5, -0.5, 1.0, -1.0],
        "measured_final_heading_deg": [90.5, -90.5, 181.0, -181.0],
        "model_wheel_radius_mm": [16.0, 16.0, 16.0, 16.0],
        "model_wheel_separation_mm": [90.0, 90.0, 90.0, 90.0],
        "surface": ["example surface"] * 4,
        "data_use": ["calibration", "calibration", "calibration", "held-back"],
    })
else:
    physical_measurements = pd.DataFrame({
        "trial_num": [1, 2],
        "measured_start_heading_deg": [np.nan, np.nan],
        "measured_final_heading_deg": [np.nan, np.nan],
        "model_wheel_radius_mm": [np.nan, np.nan],
        "model_wheel_separation_mm": [np.nan, np.nan],
        "surface": ["", ""],
        "data_use": ["calibration", "held-back"],
    })

comparison = trial_summary.merge(physical_measurements, on="trial_num")
comparison["measured_angle_deg"] = (
    comparison["measured_final_heading_deg"]
    - comparison["measured_start_heading_deg"]
)
comparison


## 10. Calculate a separation estimate for each calibration trial

Exercise 3 gives the calculation `new separation = old separation × reported angle ÷ measured angle`. The cell applies it only to rows you labelled `calibration`, then shows the individual estimates together with their mean and median.


In [ ]:
calibration_rows = comparison.loc[
    comparison["data_use"] == "calibration"
].copy()
calibration_rows["estimated_wheel_separation_mm"] = (
    calibration_rows["model_wheel_separation_mm"]
    * calibration_rows["reported_angle_deg"]
    / calibration_rows["measured_angle_deg"]
)

separation_summary = calibration_rows["estimated_wheel_separation_mm"].agg(
    ["mean", "median"]
)
display(calibration_rows[[
    "trial_num", "measured_angle_deg", "reported_angle_deg",
    "estimated_wheel_separation_mm",
]])
separation_summary


## 11. Plot measured rotation against reported rotation

Points on the dashed line indicate agreement. A consistent offset from the line suggests that the wheel-separation value used by the local model may need adjustment. Keep the held-back row out of the separation estimate and use it later to evaluate the selected value.


In [ ]:
fig, ax = plt.subplots(figsize=(6.5, 6))
sns.scatterplot(
    data=comparison,
    x="measured_angle_deg",
    y="reported_angle_deg",
    hue="trial_num",
    palette="viridis",
    s=100,
    legend=False,
    ax=ax,
)
ax.axline((0, 0), slope=1, color="black", linestyle="--")
ax.set(
    title="Physical measurement and odometry report",
    xlabel="Measured physical rotation (degrees)",
    ylabel="Reported odometry rotation (degrees)",
)
plt.show()


## What to notice

- Do the two encoder counts change with opposite signs?
- Does the odometry heading change smoothly during each turn?
- Does the reported angle tend to be larger or smaller than the physical measurement?

Save the notebook with your plots. Base conclusions about physical rotation on your independent measurements, not on the commanded angle.
